make level3 and level4 timeseris

parameterise after timeseries are working

colorbar - for side by side animation, need to hide it for single gifs and add in side-by-side animation. set a boolean flag for this.

format year

In [10]:
%matplotlib inline

import os
import sys
import datacube
import skimage.exposure
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import colors as mcolours
from IPython.display import Image
from IPython.core.display import Video

from PIL import ImageSequence
from PIL import Image as pimg

import sys

sys.path.insert(1, "../Tools/")
from dea_tools.plotting import xr_animation, rgb
from dea_tools.landcover import plot_land_cover, lc_animation, lc_colourmap

In [11]:
dc = datacube.Datacube(app="lc3_vs_lc2")

In [12]:
"""
Parameter cell - add things here that are needed to run the rest of the notebook. This way the rest of the notebook can be left alone and shouldn't need to be edited.

Try to parameterise the name of the gifs too so they are fulling either from geojson filenames or similar
"""
lon = 145.30955
lat = -37.5459

buffer = 0.2

lat_range = (lat - buffer, lat + buffer)
lon_range = (lon - buffer, lon + buffer)

output_dir = 'output_gifs'
output_gif1 = 'test_animation_1'
output_gif2 = 'test_animation_2'

interval=700

text_size = {'fontsize': 15}

# Query params
bands = ['level3', 'level4']
time = ('2015', '2020')

In [13]:
lvl3_colour = lc_colourmap('level3', colour_bar=False)
lvl4_colour = lc_colourmap('level4', colour_bar=False)

In [14]:
# use this cell to import locations for animations. e.g. geopandas points.

In [15]:
# dc_measurements = dc.list_measurements()
# dc_measurements.loc[['ga_ls_landcover_class_cyear_2']]

In [16]:
# dc_measurements.loc[['ga_ls_landcover_class_cyear_3']]

In [17]:
# Datacube loading for collection 2
query_col2 = {
    "x": x,
    "y": y,
    "time": time,
    "measurements": bands,
}

# Load available satellite data from Landsat 8 geomedian product
ds_col2 = dc.load(product="ga_ls_landcover_class_cyear_2", **query_col2)

In [18]:
# Datacube loading for collection 3
query_col3 = {
    "x": x,
    "y": y,
    "time": time,
    "measurements": bands,
}

# Load available satellite data from Landsat 8 geomedian product
ds_col3 = dc.load(product="ga_ls_landcover_class_cyear_3", **query_col3)

In [19]:
def generate_single_animations():
    #generate level 3 collection 2
    lc_animation(ds_col2.level3,
            file_name=f'{output_dir}/lc_collection2_lvl3_colorbar_{output_gif1}',
            colour_bar=True,
            label_ax=False,
            animation_interval=interval,
            width_pixels=14,
            font_size=30,
            dpi=80)
    
    #generate level 3 collection 2
    lc_animation(ds_col3.level3,
            file_name=f'{output_dir}/lc_collection3_lvl3_colorbar_{output_gif1}',
            colour_bar=True,
            label_ax=False,
            animation_interval=interval,
            width_pixels=14,
            font_size=30,
            dpi=80)
    # Gwenerate the level 3 animations WITHOUT the colorbar - for the pairwise animations
    lc_animation(ds_col2.level3,
            file_name=f'{output_dir}/lc_collection2_lvl3_{output_gif1}',
            colour_bar=False,
            label_ax=False,
            animation_interval=interval,
            width_pixels=14,
            font_size=30,
            dpi=80)
    

    lc_animation(ds_col3.level3,
            file_name=f'{output_dir}/lc_collection3_lvl3_{output_gif1}',
            colour_bar=False,
            label_ax=False,
            animation_interval=interval,
            width_pixels=14,
            font_size=30,
            dpi=80)
    
    #generate level 3 collection 2
    lc_animation(ds_col2.level4,
            file_name=f'{output_dir}/lc_collection2_lvl4_{output_gif1}',
            colour_bar=False,
            label_ax=False,
            animation_interval=interval,
            width_pixels=14,
            font_size=30,
            dpi=80)
    
    #generate level 3 collection 2
    lc_animation(ds_col3.level4,
            file_name=f'{output_dir}/lc_collection3_lvl4_{output_gif1}',
            colour_bar=False,
            label_ax=False,
            animation_interval=interval,
            width_pixels=14,
            font_size=30,
            dpi=80)

In [20]:
generate_single_animations()

In [21]:
def generate_level3_pairwise_animation(gif_filename1, gif_filename2):
    # open the files
    fname1 = f'{output_dir}/lc_collection2_lvl3_{output_gif1}.gif'
    fname2 = f'{output_dir}/lc_collection3_lvl3_{output_gif1}.gif'
    
    gif1 = pimg.open(fname1)
    gif2 = pimg.open(fname2)
    
    frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif1)]
    frames2 = [frame.copy() for frame in ImageSequence.Iterator(gif2)]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    plt.subplots_adjust(wspace=0.05)

    # make sure gif animations are synced
    def frame_update(frame):
        ax1.clear()
        ax2.clear()
        ax1.imshow(frames1[frame % len(frames1)])
        ax2.imshow(frames2[frame % len(frames2)])
        ax1.axis('off')
        ax2.axis('off')


    # # animation object
    animation_object = animation.FuncAnimation(fig, 
                                               frame_update, 
                                               frames=len(frames1),  
                                               interval=500)

    pairwise_fname = f'{output_dir}/lc_collection_2_3_lvl3_test_combo_animation.gif'
    animation_object.save(pairwise_fname, writer="imagemagick")
    plt.close()

In [22]:
def generate_level4_pairwise_animation(gif_filename1, gif_filename2):
    # open the files
    fname1 = f'{output_dir}/lc_collection2_lvl4_{output_gif1}.gif'
    fname2 = f'{output_dir}/lc_collection3_lvl4_{output_gif1}.gif'
    
    gif1 = pimg.open(fname1)
    gif2 = pimg.open(fname2)
    
    frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif1)]
    frames2 = [frame.copy() for frame in ImageSequence.Iterator(gif2)]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    plt.subplots_adjust(wspace=0.05)

    # make sure gif animations are synced
    def frame_update(frame):
        ax1.clear()
        ax2.clear()
        ax1.imshow(frames1[frame % len(frames1)])
        ax2.imshow(frames2[frame % len(frames2)])
        ax1.axis('off')
        ax2.axis('off')


    # # animation object
    animation_object = animation.FuncAnimation(fig, 
                                               frame_update, 
                                               frames=len(frames1),  
                                               interval=500)

    pairwise_fname = f'{output_dir}/lc_collection_2_3_lvl4_test_combo_animation.gif'
    animation_object.save(pairwise_fname, writer="imagemagick")
    plt.close()

In [23]:
generate_level3_pairwise_animation(output_gif1, output_gif2)

MovieWriter imagemagick unavailable; using Pillow instead.


In [24]:
generate_level4_pairwise_animation(output_gif1, output_gif2)

MovieWriter imagemagick unavailable; using Pillow instead.


In [25]:
# add text to gif after gif creation - test

gif1 = pimg.open('JAG_notebooks/landcover_collections_animations/output_gifs/lc_collection2_lvl3_test_animation_1.gif')
frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif1)]

text = "test"

In [26]:
# # get two animations side  by side

# gif1 = pimg.open(output_dir_gif1)
# gif2 = pimg.open(output_dir_gif2)

# # Ensure the images are opened as GIFs
# if not gif1.is_animated or not gif2.is_animated:
#     raise ValueError("Both files must be animated GIFs")

In [27]:
# # Extract frames from the GIFs
# frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif1)]
# frames2 = [frame.copy() for frame in ImageSequence.Iterator(gif2)]

In [28]:
# fig, (ax1, ax2) = plt.subplots(1,2)

# #function to make sure gif animations are synced
# def frame_update(frame):
#     ax1.clear()
#     ax2.clear()
#     ax1.imshow(frames1[frame % len(frames1)])
#     ax2.imshow(frames2[frame % len(frames2)])
#     ax1.axis('off')
#     ax2.axis('off')
    
    
# # animation object
# animation_object = animation.FuncAnimation(fig, 
#                                            frame_update, 
#                                            frames=len(frames1),  
#                                            interval=500)

# animation_object.save("animation_col_comparison_outputs/test_combo_animation.gif", writer="imagemagick")

# plt.close()

# Image("animation_col_comparison_outputs/test_combo_animation.gif", embed=True)